In [1]:
# Run this in 01_data_exploration.ipynb — Cell 1
import sys, shutil, os
sys.path.append("..")

# Clean old indexes
for path in [
    "../data/processed/chunks.json",
    "../data/processed/manifest.json",
    "../vector_rag/chroma_db",
    "../vectorless_rag/bm25_index.pkl",
    "../vectorless_rag/bm25_manifest.json"
]:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print(f"🗑️  {path}")
    elif os.path.exists(path):
        os.remove(path)
        print(f"🗑️  {path}")

print("\n✅ Clean — ready to rebuild")

🗑️  ../data/processed/chunks.json
🗑️  ../data/processed/manifest.json
🗑️  ../vector_rag/chroma_db

✅ Clean — ready to rebuild


In [2]:
# Cell 2 — Rebuild with new architecture
from data_loader             import run_preprocessing_pipeline
from vector_rag.indexer      import index_chunks
from vectorless_rag.indexer  import build_bm25_index

data = run_preprocessing_pipeline()

print(f"Parents  : {len(data['parents'])}")
print(f"Children : {len(data['children'])}")

index_chunks(data)
build_bm25_index(data)

print("\n🎉 All indexes rebuilt with improved architecture!")

   PHASE 2 — SCALABLE PREPROCESSING PIPELINE
  🆕 New PDF detected: amazon_10k.pdf
  🆕 New PDF detected: microsoft_10k.pdf
  🆕 New PDF detected: netflix_10k.pdf
  🆕 New PDF detected: nvidia_10k.pdf

📂 Processing 4 new PDF(s)...



Processing PDFs:  25%|██▌       | 1/4 [00:00<00:01,  2.21it/s]

   ✅ AMAZON_10K: 90 pages → 410 parents, 1468 children


Processing PDFs:  50%|█████     | 2/4 [00:03<00:03,  1.90s/it]

   ✅ MICROSOFT_10K: 156 pages → 629 parents, 2242 children


...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


📤 Indexing 7913 child chunks...
   Embedding model: BAAI/bge-large-en-v1.5


Embedding children: 100%|██████████| 124/124 [32:02<00:00, 15.50s/it]


✅ ChromaDB updated — 7913 total vectors

   VECTORLESS RAG INDEXER — BM25 + Financial Tokenizer

✅ BM25 already up to date — 7913 children indexed


🎉 All indexes rebuilt with improved architecture!


In [3]:
# Cell 3 — Quick retrieval test
from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

q = "What was NVIDIA's total revenue?"

r1 = vec.ask(q)
r2 = vl.ask(q)

vec.show(r1)
vl.show(r2)

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 7913 child vectors
Mistral client ready - model: mistral-medium-latest
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents

🔁 Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Reranker ready

  METHOD  : Vector RAG — BGE + HNSW + Parent-Child
  Q: What was NVIDIA's total revenue?
───────────────────────────────────────────────────────
  A: For **NVIDIA Corporation**, the total revenue was:
- **$215,938 million** for the fiscal year ended **January 25, 2026**,
- **$130,497 million** for the fiscal year ended **January 26, 2025**,
- **$60,922 million** for the fiscal year ended **January 28, 2024**.
───────────────────────────────────────────────────────
   • NVIDIA | Page 69 | Score 7.3471
   • NVIDIA | Page 51 | Score 6.6990
   • NVIDIA | Page 79 | Score 5.4078
   • NVIDIA | Page 55 | Score 4.0247
   • NVIDIA | Page 52 | Score 3.9045
───────────────────────────────────────────────────────
  Retrieval: 0.4212s | Rerank: 7.8007s | Generation: 2.8780s | Total: 11.0999s


  METHOD  : Vectorless RAG — BM25 + Financial Tokenizer
  Q: What was NVIDIA's total revenue?
───────────────────────────────────────────────────────
  A: For **NVIDIA**, the total revenue fo

In [4]:
import traceback
from vectorless_rag.pipeline import VectorlessRAGPipeline

try:
    vl = VectorlessRAGPipeline()
    r2 = vl.ask("What was NVIDIA's total revenue?")
    print("Answer:", r2)
    vl.show(r2)
except Exception as e:
    print("FULL ERROR:")
    traceback.print_exc()
    print("\nException details:", str(e))

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents

Answer: {'question': "What was NVIDIA's total revenue?", 'answer': 'For **NVIDIA**, the total revenue for the fiscal year ended **January 25, 2026**, was **$139,297 million** (as stated in Source 3). For the fiscal year ended **January 26, 2025**, total revenue was **$87,960 million**.', 'retrieved': [{'text': 'of total revenue, all of which were primarily attributable to the Compute & Networking segment.\nFor fiscal year 2025, sales to one direct customer represented 12% of total revenue and sales to two direct customers each represented 11% of total revenue, all\nof which were primarily attributable to the Compute & Networking segment.\nFor fiscal year 2024, sales to one direct customer represented 13% of total revenue, and were primarily attributable to the Compute & Networking segment.\nIndirect Customers – Indire

In [5]:
from data_loader import run_preprocessing_pipeline
from vector_rag.indexer import index_chunks
from vectorless_rag.indexer import build_bm25_index

data = run_preprocessing_pipeline()
print(f"Parents: {len(data['parents'])} | Children: {len(data['children'])}")

index_chunks(data)      # BGE model downloads ~438MB on first run — wait for it
build_bm25_index(data)
print("🎉 Done")

   PHASE 2 — SCALABLE PREPROCESSING PIPELINE
  ✅ Already processed: amazon_10k.pdf — skipping
  ✅ Already processed: microsoft_10k.pdf — skipping
  ✅ Already processed: netflix_10k.pdf — skipping
  ✅ Already processed: nvidia_10k.pdf — skipping

✅ Nothing new to process.
   Parents : 2010
   Children: 7913

Parents: 2010 | Children: 7913
   VECTOR RAG INDEXER — Parent-Child + BGE + HNSW

✅ ChromaDB already up to date — 7913 vectors

   VECTORLESS RAG INDEXER — BM25 + Financial Tokenizer

✅ BM25 already up to date — 7913 children indexed

🎉 Done


In [6]:
from vector_rag.pipeline import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline
from hybrid_rag.pipeline import HybridRAGPipeline

vec    = VectorRAGPipeline()
vl     = VectorlessRAGPipeline()
hybrid = HybridRAGPipeline()

q = "What was Amazon Web Services revenue for the most recent fiscal year?"
print("[VECTOR]    ", vec.ask(q)["answer"][:200])
print("[VECTORLESS]", vl.ask(q)["answer"][:200])
print("[HYBRID]    ", hybrid.ask(q)["answer"][:200])

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 7913 child vectors
Mistral client already initialised - reusing
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents

🔧 Initialising Hybrid RAG Pipeline...
✅ ChromaDB loaded — 7913 child vectors
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Hybrid RAG ready — 7913 vectors | 7913 BM25 children | 2010 parents

[VECTOR]     This information was not found in the retrieved sections. The context mentions **Amazon's (AWS) revenue increased 20% YoY in 2025** but does not provide the exact revenue figure for the most recent fi
[VECTORLESS] This information was not found in the retrieved sections. The context mentions **Amazon Web Services (AWS) revenue increased 20% YoY in 2025** but does not provide the exact revenue figure for the mos
[HYBRID]   

In [7]:
from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline

vec = VectorRAGPipeline()
vl  = VectorlessRAGPipeline()

q = "What is the combined revenue of Microsoft and Amazon?"

r1 = vec.ask(q)
r2 = vl.ask(q)

vec.show(r1)
vl.show(r2)

🔧 Initialising Vector RAG Pipeline...
✅ ChromaDB loaded — 7913 child vectors
Mistral client already initialised - reusing
✅ Vector RAG ready — 2010 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 7913 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 7913 children, 2010 parents


  METHOD  : Vector RAG — BGE + HNSW + Parent-Child
  Q: What is the combined revenue of Microsoft and Amazon?
───────────────────────────────────────────────────────
  A: This information was not found in the retrieved sections. The provided context only includes revenue figures for **Microsoft** ($281.7 billion in fiscal 2025, $245.1 billion in 2024, and $211.9 billion in 2023) and does not mention **Amazon**.
───────────────────────────────────────────────────────
   • MICROSOFT | Page 85 | Score 0.7501
   • MICROSOFT | Page 39 | Score 0.0718
   • MICROSOFT | Page 35 | Score -0.1599
   • MICROSOFT | Page 39 | Score -0.2394
   • MICROSOFT | Page 39 | S